# RCIS2 trimmed parser and preprocessor 

This version fixes the header parsing bug.


In [ ]:
import gzip
import re
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 120)

INPUT_FILE = "full_260129.rci.txt.gz"
OUTPUT_DIR = Path("processed_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def resolve_input_path(input_path: str) -> Path:
    candidates = [
        Path(input_path),
        Path.cwd() / input_path,
        Path("/mnt/data") / input_path,
    ]
    for path in candidates:
        if path.exists():
            return path.resolve()
    raise FileNotFoundError(
        f"Could not find input file '{input_path}'. Tried: {', '.join(str(p) for p in candidates)}"
    )

INPUT_PATH = resolve_input_path(INPUT_FILE)
INPUT_PATH, OUTPUT_DIR


(PosixPath('/home/akshay-paliwal/Documents/VUB/thesis_ai/data/full_260129.rci.txt.gz'),
 PosixPath('processed_rcis2_trimmed'))

## 1. Helper functions

We only keep a small set of required header fields, so the parsing is explicit.


In [2]:
def clean_header_value(text: str) -> str:
    return text.strip().lstrip(":").strip()


def extract_named_header_value(line: str, prefix: str) -> str:
    # Example:
    # line   = '# MOLECULE RNase_H1'
    # prefix = '# MOLECULE'
    # -> 'RNase_H1'
    return clean_header_value(line[len(prefix):])


def make_protein_id(protein_name: str, swissprot: str, bmrb_id: str, entry_idx: int) -> str:
    base = swissprot or protein_name or bmrb_id or f"protein_{entry_idx}"
    base = str(base).strip()
    base = re.sub(r"\s+", "_", base)
    base = re.sub(r"[^A-Za-z0-9_.|+-]", "_", base)
    return f"entry_{entry_idx:05d}"


## 2. Optional raw preview

This helps verify what the header lines actually look like in the file.


In [3]:
with gzip.open(INPUT_PATH, "rt", encoding="utf-8", errors="replace") as f:
    raw_preview = [next(f).rstrip("\n") for _ in range(40)]

for line in raw_preview:
    print(line)


# MOLECULE         RNase_H1
# BMRB_ID          bmr4012
# SWISSPROT        A7ZHV1 2.53e-108
# PDB_CODE         1f21 (diffraction)
#
# TEMPERATURE      298.0
# pH               5.5
# IONIC_STRENGTH   n/a
# PHYSICAL_STATE   native
# SAMPLE_STATES    solution
#
# LABELLING     13C (uniform), 15N (uniform) (original '[U-99% 13C; U-99% 15N]')
# SAMPLE        sample_one: RNase_H1 (2.0mM)
#
# BMRB_ORIG:   MLKQVEIFTDGSALGNPGPGGYGAILRYRGREKTFSAGYTRTTNNRMELMAAIVALEALKEHAEVILSTDSQYVRQGITQWIHNWKKRGWKTADKKPVKNVDLWQRLDAALGQHQIKWEWVKGHAGHPENERADELARAAAMNPTLEDTGYQVEV
# STRIDE_CONS: --CCEEEEEEEEECTTTEEEEEEEEEEETTEEEEEEEEEEEECHHHHHHHHHHHHHHHCCCCEEEEEEECCHHHHHHHHHHHHHHHHHTTBTTTTCBTTTHHHHHHHHHHHHCEEEEEEECCCTTTTHHHHHHHHHHHHHHHCCCBCTTTTCC--
# STRIDE_ALL:  --CCEEEEEEEEECTTTEEEEEEEEEEETTEEEEEEEEEEEECHHHHHHHHHHHHHHHCCCCEEEEEEECCHHHHHHHHHHHHHHHHHTTBTTTTCBTTTHHHHHHHHHHHHCEEEEEEECCCTTTTHHHHHHHHHHHHHHHCCCBCTTTTCC--

#
# DATA VALUES FOR SHIFT LIST assigned_chemical_shifts_one
# seqIndex seqLabel rci rciS2 rciSc d2dH

## 3. Parse only the needed header fields and core residue fields

This is the main fix.

Instead of building header columns from arbitrary raw text, we explicitly capture only:
- `molecule`
- `bmrb_id`
- `swissprot`
- `stride_cons`
- `stride_all`

That guarantees the later code gets the exact columns it expects.


In [4]:
residue_records = []
header_records = []

current_header = None
current_entry_idx = 0
current_protein_id = None

with gzip.open(INPUT_PATH, "rt", encoding="utf-8", errors="replace") as f:
    for raw_line in f:
        stripped = raw_line.strip()

        if not stripped:
            continue

        # ----------------------------
        # Header lines
        # ----------------------------
        if stripped.startswith("#"):
            if stripped.startswith("# MOLECULE"):
                if current_header:
                    header_records.append(current_header.copy())

                current_entry_idx += 1
                current_header = {
                    "protein_entry_index": current_entry_idx
                }

            # Ignore header lines before the first protein block
            if current_header is None:
                continue

            # Explicitly parse only fields we care about
            if stripped.startswith("# MOLECULE"):
                current_header["molecule"] = extract_named_header_value(stripped, "# MOLECULE")
            elif stripped.startswith("# BMRB_ID"):
                current_header["bmrb_id"] = extract_named_header_value(stripped, "# BMRB_ID")
            elif stripped.startswith("# SWISSPROT"):
                current_header["swissprot"] = extract_named_header_value(stripped, "# SWISSPROT")
            elif stripped.startswith("# STRIDE_CONS"):
                current_header["stride_cons"] = extract_named_header_value(stripped, "# STRIDE_CONS")
            elif stripped.startswith("# STRIDE_ALL"):
                current_header["stride_all"] = extract_named_header_value(stripped, "# STRIDE_ALL")

            protein_name = current_header.get("molecule", "")
            swissprot = current_header.get("swissprot", "")
            bmrb_id = current_header.get("bmrb_id", "")
            current_protein_id = make_protein_id(protein_name, swissprot, bmrb_id, current_entry_idx)

            current_header["protein_id"] = current_protein_id
            continue

        # ----------------------------
        # Residue/data lines
        # ----------------------------
        if current_header is None or current_protein_id is None:
            continue

        parts = stripped.split()
        if len(parts) < 4:
            continue

        try:
            residue_index = int(parts[0])
            amino_acid = parts[1]
            rci = float(parts[2])
            rci_s2 = float(parts[3])
        except ValueError:
            continue

        residue_records.append({
            "protein_id": current_protein_id,
            "protein_entry_index": current_entry_idx,
            "residue_index": residue_index,
            "amino_acid": amino_acid,
            "RCI": rci,
            "RCI_S2": rci_s2,
        })

if current_header:
    header_records.append(current_header.copy())

residue_df_raw = pd.DataFrame(residue_records)
header_df_raw = pd.DataFrame(header_records).drop_duplicates(subset=["protein_id"]).reset_index(drop=True)

print("Raw residue rows:", len(residue_df_raw))
print("Raw header rows :", len(header_df_raw))


Raw residue rows: 444304
Raw header rows : 4128


## 4. Check the parsed header table

This is the key sanity check for the bug you found.

You should now see proper columns like:
- `molecule`
- `bmrb_id`
- `swissprot`
- `stride_cons`
- `stride_all`

—not broken column names built from whole header lines.


In [5]:
header_df_raw.head(10)


,protein_entry_index,molecule,protein_id,bmrb_id,swissprot,stride_cons,stride_all
0,1,RNase_H1,entry_00001,bmr4012,A7ZHV1 2.53e-108,--CCEEEEEEEEECTTTEEEEEEEEEEETTEEEEEEEEEEEECHHHHHHHHHHHHHHHCCCCEEEEEEECCHHHHHHHHHHHHHHHHHTTBTTTTCBTTTHHHHHHHHHHHHCEEE...,--CCEEEEEEEEECTTTEEEEEEEEEEETTEEEEEEEEEEEECHHHHHHHHHHHHHHHCCCCEEEEEEECCHHHHHHHHHHHHHHHHHTTBTTTTCBTTTHHHHHHHHHHHHCEEE...
1,2,Sxl-RBD1+2,entry_00002,bmr4029,P19339 7.90e-128,None,None
2,3,bovine_pancreatic_ribonuclease_A,entry_00003,bmr4031,P61823 1.75e-86,CCCHHHHHHHHHBTTTTCCCCCCHHHHHHHHHHTTTTTTCCCEEEEECCCHHHHHGGGGCEEECTTTTCCCEEETTTTEEEEEEEETTTTBTTBTCEEEEEEEECEEEEEETTTTE...,CCCHHHHHHHHHBTTTTCCCCCCHHHHHHHHHHTTTTTTCCCEEEEECCCHHHHHGGGGCEEECTTTTCCCEEETTTTEEEEEEEETTTTBTTBTCEEEEEEEECEEEEEETTTTE...
3,4,E2-HPV,entry_00004,bmr4035,P17383 2.55e-49,None,None
4,5,Apokedarcidin,entry_00005,bmr4036,P41249 3.61e-71,CCCEEEEETTEEETTTEEEEEEEECTTTTCEEEEEEEEETTTTTEEETTTTTCCEETTTTBCCEEEEETTEEEEEETTTCCCCEEEETTTTTTEEEEEEETTTTCCCEEECEE-,----EEEE-TEEETTTEEEEEEE--TTTT--------E-TTTT----TTTTTC---TTTT---EEEEETTEEEE--TTT-CCCEEEETTTTTT-EEEE---TTT-C--E-CEE-
5,6,eCyP,entry_00006,bmr4037,P0AFL3 5.38e-118,-CCCCCEEEEEETTTEEEEECTTTTTHHHHHHHHHHHHHTTTTTTBCCCEETTTEEEECCCTTTTTCCCCCCCCCCTTTTCCCCTTTTEEECCCCTTTTTTTTEEEECCCTTTTTC...,-------EEEE-TTTEEE--C-----HHHHHHHHHHHH-TTTTTT----EETTTEE---C--TTT--CCCCCCC----------TTT-EEEC---TT------EEE-C--------...
6,7,SytI-C2A_apo,entry_00007,bmr4039,. 0.00e+00,CCCEEEEEEEEEETTTTEEEEEEEEEECCCCCBTTTBCCCEEEEEEETTTCCCEECCCTTTTTTTECCEEEEETTCGGGGGGCEEEEEEECCTTTTCCCEEEEEEEEGGGCCTTTE...,CCC-EEEE-EE-------------E----C---TTT-----EEEEE-TTT------CCTTT-TTT----------C-------EEEEEE---TTTT-CC-----------------...
7,8,AsiA,entry_00008,bmr4040,P32267 8.86e-56,-CCHHHHHHHHHHHHHHHHHCCTTTTTTTHHHHHHHHHHHCCCTTTTCCTTTHHHHHHHCCCHHHHHHHHHHCHHHHHHHHHHHHHHCC-,-CC-HHHHHHHHHHHHHHHH------------HH-------C--TT--C----HHHHHH---HHHHHHHHHH-HHHHHHHHHHHH-----
8,9,DNA_TopoI,entry_00009,bmr4045,P06612 3.90e-76,CCCTTTTTTCTTEEEEEEETTTTCCEEEEEETTTTEEEEETTTTTTTCCBCCBHHHHHHHGGGCCTTHHHHHHCTTTTTTTCCEECCEETTTTEEEEEETTTTTCCCEEEEETTTT...,--------------------------------TT------TTTTTT----CCBHHHHHH-GGG----------C---TTTTCC----EETTTTEEEEE-----------EE-TTTT...
9,10,dHSF,entry_00010,bmr4046,P22813 1.82e-88,None,None


## 5. Clean and trim the residue table

This step:
- merges useful protein metadata
- filters to canonical amino acids
- sorts residues within proteins
- creates `residue_order_0`
- trims to the final residue schema


In [6]:
keep_header_cols = [
    c for c in [
        "protein_id",
        "protein_entry_index",
        "molecule",
        "swissprot",
        "bmrb_id",
        "stride_cons",
        "stride_all",
    ] if c in header_df_raw.columns
]

header_df = header_df_raw[keep_header_cols].copy().rename(columns={"molecule": "protein_name"})

residue_df = residue_df_raw.merge(
    header_df,
    on=["protein_id", "protein_entry_index"],
    how="left",
)

for col in ["protein_id", "protein_name", "swissprot", "bmrb_id", "amino_acid"]:
    if col in residue_df.columns:
        residue_df[col] = residue_df[col].fillna("").astype(str).str.strip()

canonical_aas = set("ACDEFGHIKLMNPQRSTVWY")
residue_df = residue_df[residue_df["amino_acid"].isin(canonical_aas)].reset_index(drop=True)

if residue_df.empty:
    raise ValueError("All residue rows were filtered out. Check amino acid parsing.")

residue_df = residue_df.sort_values(
    by=["protein_id", "residue_index"],
    kind="mergesort"
).reset_index(drop=True)

residue_df["residue_order_0"] = residue_df.groupby("protein_id").cumcount()

residue_keep_cols = [
    "protein_id",
    "protein_name",
    "swissprot",
    "bmrb_id",
    "residue_index",
    "residue_order_0",
    "amino_acid",
    "RCI",
    "RCI_S2",
]
residue_df = residue_df[residue_keep_cols].copy()

residue_df.head()


,protein_id,protein_name,swissprot,bmrb_id,residue_index,residue_order_0,amino_acid,RCI,RCI_S2
0,entry_00001,RNase_H1,A7ZHV1 2.53e-108,bmr4012,7,0,I,0.0286,0.8362
1,entry_00001,RNase_H1,A7ZHV1 2.53e-108,bmr4012,8,1,F,0.0281,0.8385
2,entry_00001,RNase_H1,A7ZHV1 2.53e-108,bmr4012,9,2,T,0.0268,0.8448
3,entry_00001,RNase_H1,A7ZHV1 2.53e-108,bmr4012,11,3,G,0.0206,0.8759
4,entry_00001,RNase_H1,A7ZHV1 2.53e-108,bmr4012,12,4,S,0.0211,0.8731


## 6. Build the protein manifest

This table has one row per protein and is the main input reference for Hydra.


In [7]:
manifest = residue_df.groupby("protein_id", sort=False).agg(
    protein_name=("protein_name", "first"),
    swissprot=("swissprot", "first"),
    bmrb_id=("bmrb_id", "first"),
    n_residues=("residue_index", "size"),
).reset_index()

seqs = residue_df.groupby("protein_id", sort=False)["amino_acid"].apply(
    lambda x: "".join(x.astype(str))
).reset_index(name="sequence")
manifest = manifest.merge(seqs, on="protein_id", how="left")
manifest["sequence_length"] = manifest["sequence"].str.len()

ss_cols = [c for c in ["protein_id", "stride_cons", "stride_all"] if c in header_df.columns]
if ss_cols:
    ss_df = header_df[ss_cols].drop_duplicates(subset=["protein_id"])
    manifest = manifest.merge(ss_df, on="protein_id", how="left")

manifest = manifest[[c for c in [
    "protein_id",
    "protein_name",
    "swissprot",
    "bmrb_id",
    "sequence",
    "sequence_length",
    "n_residues",
    "stride_cons",
    "stride_all",
] if c in manifest.columns]]

manifest.head()


,protein_id,protein_name,swissprot,bmrb_id,sequence,sequence_length,n_residues,stride_cons,stride_all
0,entry_00001,RNase_H1,A7ZHV1 2.53e-108,bmr4012,IFTGSALGNGGGYGAILRYRGREKTFSAGYTRTNNRMELMAAIVALEALKEHEVILSTDSQYVRQGITQWIHNWKKRGWKTADKKVKNVDLWQRLDAALGQHQIKWEWVKGHAGHE...,138,138,--CCEEEEEEEEECTTTEEEEEEEEEEETTEEEEEEEEEEEECHHHHHHHHHHHHHHHCCCCEEEEEEECCHHHHHHHHHHHHHHHHHTTBTTTTCBTTTHHHHHHHHHHHHCEEE...,--CCEEEEEEEEECTTTEEEEEEEEEEETTEEEEEEEEEEEECHHHHHHHHHHHHHHHCCCCEEEEEEECCHHHHHHHHHHHHHHHHHTTBTTTTCBTTTHHHHHHHHHHHHCEEE...
1,entry_00002,Sxl-RBD1+2,P19339 7.90e-128,bmr4029,SDDLMNDPRASNTNLIVNYLTDRYALFRAIGPINTCRDYKTGYFGYAFVFTSEMDSQRAIKVLNGITVRNKKVSYARPGGESIKDTNLYVTNLPRTITDDQLDTIFGKYGSIVKNI...,168,168,None,None
2,entry_00003,bovine_pancreatic_ribonuclease_A,P61823 1.75e-86,bmr4031,KETAAAKFERQHMDSSTSAASSSNYCNQMMKSRNLTKDRCKPVNTFVHESLADVQAVCSQKNVACKNGQTNCYQSYSTMSITDCRETGSSKYPNCAYKTTQANKHIIVACEGNPYV...,124,124,CCCHHHHHHHHHBTTTTCCCCCCHHHHHHHHHHTTTTTTCCCEEEEECCCHHHHHGGGGCEEECTTTTCCCEEETTTTEEEEEEEETTTTBTTBTCEEEEEEEECEEEEEETTTTE...,CCCHHHHHHHHHBTTTTCCCCCCHHHHHHHHHHTTTTTTCCCEEEEECCCHHHHHGGGGCEEECTTTTCCCEEETTTTEEEEEEEETTTTBTTBTCEEEEEEEECEEEEEETTTTE...
3,entry_00004,E2-HPV,P17383 2.55e-49,bmr4035,MATTPIIHLKGDANILKCLRYRLSKYKQLYEQVSSTWHWTCTDGKHKNAIVTLTYISTSQRDDFLNTVKIPNTVSVSTGYMTI,83,83,None,None
4,entry_00005,Apokedarcidin,P41249 3.61e-71,bmr4036,ASAAVSVSPATGLADGATVTVSASGFATSTSATALQCAILADGRGACNVAEFHDFSLSGGEGTTSVVVRRSFTGYVMPDGPEVGAVDCDTAPGGCEIVVGGNTGEYGNAAISFG,114,114,CCCEEEEETTEEETTTEEEEEEEECTTTTCEEEEEEEEETTTTTEEETTTTTCCEETTTTBCCEEEEETTEEEEEETTTCCCCEEEETTTTTTEEEEEEETTTTCCCEEECEE-,----EEEE-TEEETTTEEEEEEE--TTTT--------E-TTTT----TTTTTC---TTTT---EEEEETTEEEE--TTT-CCCEEEETTTTTT-EEEE---TTT-C--E-CEE-


## 7. Sanity checks

These are good to verify before moving to Hydra.


In [8]:
summary = {
    "n_proteins": manifest["protein_id"].nunique(),
    "n_residues": len(residue_df),
    "mean_RCI_S2": residue_df["RCI_S2"].mean(),
    "median_RCI_S2": residue_df["RCI_S2"].median(),
    "min_RCI_S2": residue_df["RCI_S2"].min(),
    "max_RCI_S2": residue_df["RCI_S2"].max(),
}
pd.Series(summary)


n_proteins         4089.000000
n_residues       444304.000000
mean_RCI_S2           0.756963
median_RCI_S2         0.830500
min_RCI_S2            0.061700
max_RCI_S2            0.999200
dtype: float64

In [9]:
(manifest["sequence_length"] == manifest["n_residues"]).value_counts(dropna=False)


True    4089
Name: count, dtype: int64

## 8. Save final outputs

The FASTA file is the one you will feed directly to the PLMs on Hydra.


In [10]:
residue_path = OUTPUT_DIR / "residue_table_trimmed.csv"
manifest_path = OUTPUT_DIR / "protein_manifest_trimmed.csv"
fasta_path = OUTPUT_DIR / "sequences_trimmed.fasta"

residue_df.to_csv(residue_path, index=False)
manifest.to_csv(manifest_path, index=False)

with open(fasta_path, "w", encoding="utf-8") as f:
    for _, row in manifest.iterrows():
        f.write(f">{row['protein_id']}\n")
        f.write(f"{row['sequence']}\n")

print(residue_path)
print(manifest_path)
print(fasta_path)


processed_rcis2_trimmed/residue_table_trimmed.csv
processed_rcis2_trimmed/protein_manifest_trimmed.csv
processed_rcis2_trimmed/sequences_trimmed.fasta
